## Baseline scores on semart using CLIP

In [1]:
%load_ext autoreload
%autoreload 2

import torch.nn.functional as F

import open_clip 
import numpy as np
import pandas as pd


from src.model import SheafMultimodalGNN
from src.utils import *
from src.data import *
# from src.ClusterData import ClusterData, ClusterLoader
from torch_geometric.data import DataLoader
from src.metrics import *

triplets = '../artistic_sheaf/data/triplets_semart_test_orig.json'
#triplets = '../artistic_sheaf/data/full_triplets.json'
loaded_data = load_json_data(triplets)#[:9914]
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 4569 triplets from ../artistic_sheaf/data/triplets_semart_test_orig.json


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
print(f"Using device: {device}")
seed_everything(seed=42)

# Load tokenizer and preprocessing
tokenizer = open_clip.get_tokenizer('ViT-B-32')
_, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
# Initialize the model
model = SheafMultimodalGNN(
    latent_dim=512,
    edge_attr_dim=512,
    num_layers=3,
    step_size=1.0,
    lr=1e-4,
    test=False,
    clip_grad=True,
    device='cuda' if torch.cuda.is_available() else 'mps'
)
    
# Load checkpoint
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=43-val_loss=5.36.ckpt", map_location=device) # sheafCLIP frozen CLIP
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=23-val_loss=5.41.ckpt", map_location=device) # no sheaf frozen CLIP
checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=16-val_loss=5.36.ckpt", map_location=device) # sheafCLIP CLIP grad
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)
model.eval()
print()

Using device: mps



### Testing on normal dataset

In [ ]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data, preprocess, tokenizer, base_folder='../SemArt/', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset) // 3, shuffle=False)

4569


In [ ]:
#clip_texts = get_clip_texts(loaded_data, 'item2', get_tokenizer('ViT-B-32'), model)
clip_images = []
clip_texts = []
for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings, _ = model(x_img, x_text, edge_index, edge_attr)
        clip_images.append(F.normalize(embeddings[: len(edge_attr), :], dim=1))
        clip_texts.append(F.normalize(embeddings[len(edge_attr):, :], dim=1))
        
clip_images = torch.cat(clip_images, dim=0)
clip_texts = torch.cat(clip_texts, dim=0)

print(f"Extracted {len(clip_texts)} text embeddings, each of shape {clip_texts[0].shape}")
print(f"Extracted {len(clip_images)} image embeddings, each of shape {clip_images[0].shape}")

clip_images = clip_images.cpu().detach().numpy()
clip_texts = clip_texts.cpu().detach().numpy()

In [ ]:
# # save embeddings images and test

# np.save('data/clip_images_grad_clip.npy', clip_images)
# np.save('data/clip_texts_grad_clip.npy', clip_texts)


In [ ]:
# clip_images = np.load('data/clip_images_grad_clip.npy')
# clip_texts = np.load('data/clip_texts_grad_clip.npy')

In [11]:
# take a subset of the image embeddings and plot them with plotly interactively (in 2D using umap) showing the edge_index[0, i] on hover
import umap
import plotly.express as px
reducer = umap.UMAP()
#from sklearn.decomposition import PCA
#reducer = PCA(n_components=2)

embedding_2d = reducer.fit_transform(np.concatenate([clip_images[: 400], clip_texts[: 400]], axis=0) ) # take only first 
fig = px.scatter(x=embedding_2d[:, 0], y=embedding_2d[:, 1],
                hover_data=[np.concatenate([np.arange(400), np.arange(400)], axis=0),
                            np.concatenate([test_graph_data.edge_index[0, :400].cpu().numpy(), test_graph_data.edge_index[1, :400].cpu().numpy()], axis=0)], 
                color=['images']*400 + ['text']*400)
fig.show()

### Image-to-text retrieval	and Text-to-image retrieval		
r@1	r@5	r@10	

In [ ]:
loaded_data = load_json_data("data/triplets_semart_test_orig.json")#[:9914]
print(f"Loaded {len(loaded_data)} triplets from data/triplets_semart_test_orig.json")

adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data)
print(f"Adjacency matrix shape: {adj_matrix.shape}")

sim_matrix = get_sim_matrix([t["item1"] for t in loaded_data], 
                            [t["item2"] for t in loaded_data], 
                            clip_images, clip_texts,
                            img_to_idx, txt_to_idx)
print(f"Similarity matrix shape: {sim_matrix.shape}")

Loaded 4569 triplets from data/triplets_semart_test_orig.json


In [ ]:
compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])

### Retrieval per type of relationship

In [ ]:
verbose = False
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    #print(f"Loaded {len(loaded_data_new)} triplets for type {typ}")
    adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new)
    #print(f"Adjacency matrix shape: {adj_matrix.shape}")
    sim_matrix = get_sim_matrix([t["item1"] for t in loaded_data_new], 
                            [t["item2"] for t in loaded_data_new], 
                            clip_images, clip_texts,
                            img_to_idx, txt_to_idx)
    #print(f"Similarity matrix shape: {sim_matrix.shape}")
    print(compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10]))

    if verbose:
        # Print which query gives which recommendation (text or image path)
        # For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=5)

        query_field = 'item1'  # image path
        rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}

        for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
            query = loaded_data_new[i][query_field]
            recommendations = [idx_to_txt[j] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {loaded_data_new[i][rec_field]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

### Zero shot classification

In [4]:
annotations = '../SemArt/semart_test.csv'
df = pd.read_csv(annotations, sep='\t', encoding='latin1')
df.shape

(1069, 9)

In [5]:
loaded_data_new = []
for itm in df['IMAGE_FILE']:
    loaded_data_new.append({})
    loaded_data_new[-1]['item1'] = 'Images/' + itm
    author = df[df['IMAGE_FILE'] == itm]['AUTHOR'].values[0]
    loaded_data_new[-1]['author'] = f"Artwork by {author}"
    timeframe = df[df['IMAGE_FILE'] == itm]['TIMEFRAME'].values[0]
    loaded_data_new[-1]['timeframe'] = f"Artwork painted in {timeframe}"
    school = df[df['IMAGE_FILE'] == itm]['SCHOOL'].values[0]
    loaded_data_new[-1]['school'] = f"Artwork from the {school} school"
    material = df[df['IMAGE_FILE'] == itm]['TECHNIQUE'].values[0].split(',')[0]
    loaded_data_new[-1]['material'] = f"Artwork made with {material}"
    genre = df[df['IMAGE_FILE'] == itm]['TYPE'].values[0]
    loaded_data_new[-1]['genre'] = f"Artwork of the {genre} genre"
    loaded_data_new[-1]['link'] = 'metadata'

print(f"Updated loaded_data with authors, total items: {len(loaded_data_new)}")

Updated loaded_data with authors, total items: 1069


In [ ]:
def get_zero_shot_accuracy(item2, verbose=False):
    test_graph_data, _, _ = build_graph_from_json(loaded_data_new, preprocess, tokenizer, 
                                                                           base_folder='../SemArt/', item2=item2)
    test_graph_data = test_graph_data.to(device)
    test_dataset = GraphEdgeDataset(test_graph_data, device=device)
    test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

    for batch in test_loader:
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'test')
        embeddings, _ = model(x_img, x_text, edge_index, edge_attr)
        clip_images_cls = F.normalize(embeddings[: len(edge_attr), :], dim=1).cpu().detach().numpy()
        clip_texts_author = F.normalize(embeddings[len(edge_attr): , :], dim=1).cpu().detach().numpy()

        print(f"Extracted {len(clip_texts_author)} text embeddings, each of shape {clip_texts_author[0].shape}")

    adj_matrix_author, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new, field=item2)
    sim_matrix_author = get_sim_matrix([t["item1"] for t in loaded_data_new], 
                                [t[item2] for t in loaded_data_new], 
                                clip_images_cls, clip_texts_author,
                                img_to_idx, txt_to_idx)

      
    if verbose:
        # print(f"Adjacency matrix shape: {adj_matrix_author.shape}", 
        #     f"Similarity matrix shape: {sim_matrix_author.shape}")
        # Print which query gives which recommendation (text or image path)
        # For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix_author), k=5)

        query_field = 'item1'  # image path
        rec_field = item2      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}

        for i, rec_indices in enumerate(recs):
            query = loaded_data_new[i][query_field]
            recommendations = [idx_to_txt[j] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {loaded_data_new[i][rec_field]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

    return compute_image_to_text_accuracy(sim_matrix_author, adj_matrix_author)
    

In [7]:
for item2 in ['author', 'timeframe', 'school', 'material', 'genre']:
    print(f"Zero-shot classification accuracy for {item2}:")
    acc = get_zero_shot_accuracy(item2)
    print(acc)
    print()

Zero-shot classification accuracy for author:


100%|██████████| 1069/1069 [00:26<00:00, 40.44it/s]
/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


Extracted 1069 text embeddings, each of shape (512,)
0.07951356407857811

Zero-shot classification accuracy for timeframe:


100%|██████████| 1069/1069 [00:20<00:00, 51.25it/s]


Extracted 1069 text embeddings, each of shape (512,)
0.13096351730589337

Zero-shot classification accuracy for school:


...

Extracted 1069 text embeddings, each of shape (512,)
0.11973807296538821

Zero-shot classification accuracy for material:


100%|██████████| 1069/1069 [00:18<00:00, 58.34it/s]


Extracted 1069 text embeddings, each of shape (512,)
0.1580916744621141

Zero-shot classification accuracy for genre:


100%|██████████| 1069/1069 [00:17<00:00, 62.71it/s]


Extracted 1069 text embeddings, each of shape (512,)
0.19457436856875585



## Evaluation with pseudo edge index

In [3]:
triplets = '../artistic_sheaf/data/full_triplets.json'
loaded_data = load_json_data(triplets)#[:9914]
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

Loaded 9138 triplets from ../artistic_sheaf/data/full_triplets.json


In [4]:
# subselect elts in loaded_data where source == generated_i2t
loaded_data_i2t = [elt for elt in loaded_data if elt['source'] == 'generated_i2t']
print(f"Loaded {len(loaded_data_i2t)} triplets from {triplets} with source generated_i2t")
loaded_data_t2i = [elt for elt in loaded_data if elt['source'] == 'generated_t2i']
print(f"Loaded {len(loaded_data_t2i)} triplets from {triplets} with source generated_t2i")

Loaded 4569 triplets from ../artistic_sheaf/data/full_triplets.json with source generated_i2t
Loaded 4569 triplets from ../artistic_sheaf/data/full_triplets.json with source generated_t2i


In [5]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_i2t, preprocess, tokenizer, base_folder='../SemArt/', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset) // 3, shuffle=False)

100%|██████████| 4569/4569 [00:15<00:00, 286.43it/s] 


4569


/Users/ludovicaschaerf/miniforge3/envs/sheaf_arm/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [6]:
clip_images_i2t = []
for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings, _ = model(x_img, x_text, edge_index, edge_attr)
        clip_images_i2t.append(F.normalize(embeddings[: len(edge_attr), :], dim=1))
clip_images_i2t = torch.cat(clip_images_i2t, dim=0)

print(f"Extracted {len(clip_images_i2t)} image embeddings, each of shape {clip_images_i2t[0].shape}")

clip_images_i2t = clip_images_i2t.cpu().detach().numpy()

torch.Size([963, 3, 224, 224]) torch.Size([771, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
torch.Size([612, 3, 224, 224]) torch.Size([507, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
torch.Size([608, 3, 224, 224]) torch.Size([550, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
Extracted 4569 image embeddings, each of shape torch.Size([512])


In [7]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_t2i, preprocess, tokenizer, base_folder='../SemArt/', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset) // 3, shuffle=False)

100%|██████████| 4569/4569 [00:15<00:00, 303.79it/s] 


4569


In [8]:
clip_texts_t2i = []

for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings, _ = model(x_img, x_text, edge_index, edge_attr)
        clip_texts_t2i.append(F.normalize(embeddings[len(edge_attr):, :], dim=1))
clip_texts_t2i = torch.cat(clip_texts_t2i, dim=0)

print(f"Extracted {len(clip_texts_t2i)} text embeddings, each of shape {clip_texts_t2i[0].shape}")

clip_texts_t2i = clip_texts_t2i.cpu().detach().numpy()

torch.Size([684, 3, 224, 224]) torch.Size([1488, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
torch.Size([633, 3, 224, 224]) torch.Size([1520, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
torch.Size([538, 3, 224, 224]) torch.Size([1518, 77]) torch.Size([2, 1523]) torch.Size([1523, 77])
Extracted 4569 text embeddings, each of shape torch.Size([512])


In [9]:
from collections import defaultdict 
def reorder_predictions_by_link_item(
    predictions: np.ndarray,
    new_list,   # "data/full_triplets.json" (the list used to produce predictions)
    old_list,   # the new file with same links but different item2 assignments
    item = 'item2',  # which item to use for matching (default 'item2' for text predictions)
):
    """
    Reorder the text predictions to match the order of (link, item2) in the new triplet file.
    Assumes:
      - predictions_txt[i] corresponds to old_list[i]['item2'] with old_list[i]['link'].
      - Keys used for matching are (link, item2).
      - Handles duplicate (link, item2) by consuming old indices FIFO.
    """
    
    # Build mapping: (link, item2) -> queue of old indices
    pos_by_key = defaultdict(list)
    for idx, tr in enumerate(old_list):
        link = tr.get("link")
        item2 = tr.get(item)
        pos_by_key[(link, item2)].append(idx)

    print(len(pos_by_key), "unique (link,item) pairs in the old file.")
    # Build reorder indices to match new_list order
    
    reorder_indices = []
    missing = []
    for tr in new_list:
        key = (tr.get("link"), tr.get(item))
        if pos_by_key[key]:
            reorder_indices.append(pos_by_key[key].pop(0))  # consume one occurrence
        else:
            missing.append(key)

    if missing:
        # Raise for visibility; switch to a warning if partial overlap is expected.
        example = missing[:5]
        print(
            f"{len(missing)} (link,{item}) pairs in the new file were not found in the old predictions. "
            f"Examples: {example}"
        )

    # Reorder predictions
    idx_t = np.array(reorder_indices)
    print(reorder_indices[:10])
    predictions_reordered = predictions[idx_t]
    
    # print how many triplets (link, item1, item2) are the same in the old and new list by creating dictionaries
    # Build mapping: (link, item2) -> queue of old indices
    pos_by_key_all = defaultdict(list)
    for idx, tr in enumerate(old_list):
        link = tr.get("link")
        item2 = tr.get('item2')
        item1 = tr.get('item1')
        pos_by_key_all[(link, item2, item1)].append(idx)

    wrong = []
    for tr in new_list:
        key = (tr.get("link"), tr.get('item2'), tr.get('item1'))
        if pos_by_key_all[key]:
            reorder_indices.append(pos_by_key_all[key].pop(0))  # consume one occurrence
        else:
            wrong.append(key)

    print("Accuracy of preliminary matching (link, item1, item2):",
          1 - len(wrong) / len(new_list))

    return predictions_reordered


In [10]:
loaded_data = load_json_data("data/triplets_semart_test_orig.json")#[:9914]
print(f"Loaded {len(loaded_data)} triplets from data/triplets_semart_test_orig.json")

Loaded 4569 triplets from data/triplets_semart_test_orig.json


In [11]:
predictions_txt_new_order = reorder_predictions_by_link_item(
    clip_texts_t2i,
    new_list=loaded_data_t2i,  # use only the test portion of the loaded data
    old_list=loaded_data,
    item='item2'
)
print(predictions_txt_new_order.shape)
predictions_txt_new_order[:5]
    

4546 unique (link,item) pairs in the old file.
[0, 8, 12, 22, 24, 28, 32, 36, 40, 44]
Accuracy of preliminary matching (link, item1, item2): 0.24403589406872406
(4569, 512)


array([[-0.0004141 ,  0.04339724,  0.05340251, ..., -0.01926412,
         0.00413403,  0.01731746],
       [-0.04599962, -0.01825658,  0.02921375, ..., -0.06506981,
        -0.01110992,  0.05403557],
       [-0.01743414,  0.05956129, -0.05097187, ..., -0.03625222,
         0.02499957, -0.05406392],
       [-0.05788532, -0.01635166, -0.06560384, ...,  0.01230823,
        -0.03691173, -0.03298913],
       [-0.00270245, -0.0558102 ,  0.01303499, ..., -0.05708861,
        -0.04821299,  0.00346222]], shape=(5, 512), dtype=float32)

In [12]:
predictions_image_new_order = reorder_predictions_by_link_item(
    clip_images_i2t,
    new_list=loaded_data_i2t,  # use only the test portion of the loaded data
    old_list=loaded_data,
    item='item1',
)
print(predictions_image_new_order.shape)

2520 unique (link,item) pairs in the old file.
[0, 8, 12, 22, 24, 28, 32, 36, 40, 44]
Accuracy of preliminary matching (link, item1, item2): 0.15364412344057776
(4569, 512)


In [13]:
verbose = False
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    print(f"Loaded {len(loaded_data_new)} triplets for type {typ}")
    adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new)
    print(f"Adjacency matrix shape: {adj_matrix.shape}")
    sim_matrix = get_sim_matrix([t["item1"] for t in loaded_data_new], 
                            [t["item2"] for t in loaded_data_new], 
                            clip_images_i2t, clip_texts_t2i,
                            img_to_idx, txt_to_idx)
    print(f"Similarity matrix shape: {sim_matrix.shape}")
    print(compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10]))

    if verbose:
        # Print which query gives which recommendation (text or image path)
        # For zero-shot classification, queries are image paths (from loaded_data_new), recommendations are text (e.g., timeframe, author, etc.)
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=5)

        query_field = 'item1'  # image path
        rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}

        for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
            query = loaded_data_new[i][query_field]
            recommendations = [idx_to_txt[j] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {loaded_data_new[i][rec_field]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

Processing type: context
Loaded 1453 triplets for type context
Adjacency matrix shape: (634, 1447)
Similarity matrix shape: (634, 1447)
{'t2i_precision@1': tensor(0.1088), 't2i_recall@1': tensor(0.0685), 't2i_ndcg@1': tensor(0.1088), 't2i_precision@5': tensor(0.0356), 't2i_recall@5': tensor(0.1088), 't2i_ndcg@5': tensor(0.1015), 't2i_precision@10': tensor(0.0205), 't2i_recall@10': tensor(0.1237), 't2i_ndcg@10': tensor(0.1065), 'i2t_precision@1': tensor(0.0560), 'i2t_recall@1': tensor(0.0560), 'i2t_ndcg@1': tensor(0.0560), 'i2t_precision@5': tensor(0.0182), 'i2t_recall@5': tensor(0.0909), 'i2t_ndcg@5': tensor(0.0757), 'i2t_precision@10': tensor(0.0112), 'i2t_recall@10': tensor(0.1116), 'i2t_ndcg@10': tensor(0.0817), 'mean_precision@1': tensor(0.0824), 'mean_recall@1': tensor(0.0622), 'mean_ndcg@1': tensor(0.0824), 'mean_precision@5': tensor(0.0269), 'mean_recall@5': tensor(0.0998), 'mean_ndcg@5': tensor(0.0886), 'mean_precision@10': tensor(0.0159), 'mean_recall@10': tensor(0.1176), 'mea